# 🤖 Módulo 2 · Cuaderno 1 — Nexo actúa
### Del chatbot al agente: herramientas, *function calling*, ReAct y planificación

**AVE Ingeniería de Prompting y Agentes Autónomos · Universidad Ean**

Hasta ahora Nexo **habla**. Pero la cooperativa necesita que **haga cosas**: consultar el precio vigente, buscar a un asociado, calcular una liquidación exacta y registrar una solicitud de recogida. Un **agente** es un modelo de lenguaje que, además de responder, **decide qué acciones tomar** con herramientas y repite el ciclo hasta cumplir una meta.

**Al terminar podrás:**
1. Explicar la diferencia entre un chatbot y un agente.
2. Convertir funciones de Python en **herramientas** que el modelo puede usar (*function calling*).
3. Programar el bucle **ReAct** (razonar → actuar → observar) paso a paso, viendo cada decisión.
4. Hacer que Nexo **planifique** una meta de varios pasos antes de ejecutarla.

> Relación con el AVE: Módulo 2, Págs. 01 a 05.

## 0. Antes de empezar: cómo usar este cuaderno

Un **cuaderno de Colab** mezcla texto (como este) y **celdas de código** que puedes ejecutar.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ de la izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla, vuelve a ejecutar desde la sección de configuración.
- Lo que escribe el código aparece debajo de la celda: eso es la **salida**.
- Si cierras la pestaña o pasa mucho tiempo, Colab reinicia la sesión y hay que ejecutar todo otra vez (*Entorno de ejecución → Ejecutar todas*).

### Tu clave de Gemini (gratis)

Nexo usa los modelos **Gemini** de Google a través de su API. Necesitas una clave personal:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y haz clic en **Create API key**.
2. Copia la clave.
3. En Colab, abre el panel izquierdo y haz clic en el ícono de la **llave 🔑 (Secretos)**.
4. Agrega un secreto con nombre `GOOGLE_API_KEY`, pega la clave en *Valor* y activa **Acceso al cuaderno**.

> ⚠️ **Nunca escribas tu clave dentro del código** ni la compartas: si entregas o compartes el cuaderno, cualquiera podría usarla. Los *Secretos* de Colab la guardan aparte.

La capa gratuita de Gemini tiene un límite de solicitudes por minuto. Si ves un error **429**, espera un minuto: el código de este cuaderno ya reintenta solo.

## 1. Configuración

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

In [ ]:
# ── Datos (ficticios) de la Cooperativa Raíces ───────────────────────────
# Son la "fuente de verdad" de la cooperativa. Nexo no debe inventar nada que no esté aquí.
COOPERATIVA = {
    "nombre": "Cooperativa Raíces",
    "producto": "café pergamino seco",
    "precio_referencia_carga": 2_150_000,   # pesos colombianos por carga de 125 kg (se actualiza cada lunes)
    "fecha_precio": "lunes de esta semana",
    "horario_acopio": "lunes a sábado, de 6:00 a. m. a 2:00 p. m.",
}

REGLAMENTO = [
    "Art. 1. Precio de referencia: la cooperativa publica cada lunes el precio de referencia por carga de 125 kg de café pergamino seco. Nexo solo informa el precio vigente publicado.",
    "Art. 2. Calidad: el café se recibe con humedad entre 10 % y 12 %. Si la humedad supera el 12 %, se descuenta el 1 % del valor por cada punto adicional.",
    "Art. 3. Horario del centro de acopio: lunes a sábado de 6:00 a. m. a 2:00 p. m. Los domingos y festivos no hay recepción.",
    "Art. 4. Recogida en camión: solo para entregas de 10 cargas o más, en veredas con vía carreteable, y se solicita con 48 horas de anticipación.",
    "Art. 5. Pagos: los pagos se hacen por transferencia a la cuenta registrada del asociado dentro de los 3 días hábiles siguientes a la entrega.",
    "Art. 6. Aprobación de pagos: todo pago igual o superior a 5.000.000 de pesos requiere la aprobación de la analista de pagos.",
    "Art. 7. Anticipos de cosecha: un asociado puede pedir un anticipo de hasta el 30 % del valor entregado el año anterior.",
    "Art. 8. Datos personales: Nexo no comparte datos de un asociado con otra persona y nunca pide contraseñas ni claves bancarias.",
    "Art. 9. Quejas: toda queja se registra con un número de caso y la responde una persona del equipo en máximo 5 días hábiles.",
    "Art. 10. Asamblea: la asamblea general de asociados se realiza el último sábado de marzo.",
]

ASOCIADOS = {
    "A-101": {"nombre": "Aurelio Gómez", "vereda": "La Esperanza", "via_carreteable": True,  "cargas_anio_anterior": 48},
    "A-102": {"nombre": "Rosa Pineda",   "vereda": "El Diviso",    "via_carreteable": False, "cargas_anio_anterior": 22},
    "A-103": {"nombre": "Julián Ortiz",  "vereda": "Alto Bonito",  "via_carreteable": True,  "cargas_anio_anterior": 9},
}

print("Precio de referencia:", f"${COOPERATIVA['precio_referencia_carga']:,}".replace(",", "."), "por carga")
print("Artículos del reglamento:", len(REGLAMENTO), "| Asociados registrados:", len(ASOCIADOS))

## 2. Chatbot vs. agente

| | Chatbot | Agente |
|---|---|---|
| ¿Qué hace? | Responde con texto | Decide y ejecuta acciones |
| ¿De dónde saca los datos? | De lo que "recuerda" del entrenamiento o del prompt | De herramientas: bases de datos, cálculos, sistemas |
| ¿Cuántos pasos? | Uno: pregunta → respuesta | Varios: razona, actúa, observa, repite |
| Riesgo | Inventar información | Ejecutar acciones equivocadas → necesita límites |

## 3. Las herramientas de Nexo

Una **herramienta** es simplemente una **función de Python**. Lo importante es su **docstring** (el texto entre comillas triples): el modelo lo lee para decidir **cuándo** y **cómo** usarla. Por eso debe ser claro, como si le explicaras a un compañero nuevo.

In [ ]:
SOLICITUDES = []   # aquí se guardan las solicitudes registradas (en un proyecto real sería una base de datos)


def consultar_precio_referencia() -> dict:
    """Devuelve el precio de referencia vigente de la cooperativa, en pesos por carga de 125 kg."""
    return {"precio_por_carga": COOPERATIVA["precio_referencia_carga"], "publicado": COOPERATIVA["fecha_precio"]}


def buscar_asociado(codigo: str) -> dict:
    """Busca un asociado por su código (por ejemplo 'A-101') y devuelve su nombre, vereda,
    si su vereda tiene vía carreteable y las cargas que entregó el año anterior."""
    asociado = ASOCIADOS.get(codigo.upper().strip())
    return asociado if asociado else {"error": f"No existe un asociado con código {codigo}"}


def calcular_liquidacion(cargas: int, humedad: float) -> dict:
    """Calcula el valor a pagar por una entrega de café: cargas × precio vigente, con un descuento
    del 1 % por cada punto de humedad por encima de 12 % (Art. 2)."""
    precio = COOPERATIVA["precio_referencia_carga"]
    bruto = cargas * precio
    descuento_pct = max(0.0, humedad - 12) * 1
    neto = round(bruto * (1 - descuento_pct / 100))
    return {"bruto": bruto, "descuento_pct": descuento_pct, "neto": neto,
            "requiere_aprobacion": neto >= 5_000_000}   # Art. 6


def verificar_recogida(codigo: str, cargas: int) -> dict:
    """Verifica si un asociado cumple las condiciones de recogida en camión del Art. 4:
    10 cargas o más y vereda con vía carreteable."""
    asociado = ASOCIADOS.get(codigo.upper().strip())
    if not asociado:
        return {"error": "Asociado no encontrado"}
    motivos = []
    if cargas < 10:
        motivos.append(f"tiene {cargas} cargas y el mínimo es 10")
    if not asociado["via_carreteable"]:
        motivos.append(f"la vereda {asociado['vereda']} no tiene vía carreteable")
    return {"aplica": not motivos, "motivos": motivos, "anticipacion_requerida_horas": 48}


def registrar_solicitud_recogida(codigo: str, cargas: int) -> dict:
    """Registra una solicitud de recogida en camión. Úsala SOLO después de verificar que la recogida aplica."""
    numero = f"REC-{len(SOLICITUDES) + 1:03d}"
    SOLICITUDES.append({"numero": numero, "codigo": codigo, "cargas": cargas})
    return {"registrada": True, "numero": numero}


HERRAMIENTAS = [consultar_precio_referencia, buscar_asociado, calcular_liquidacion,
                verificar_recogida, registrar_solicitud_recogida]

# Probamos las funciones directamente (sin IA) para asegurarnos de que funcionan:
print(calcular_liquidacion(14, 13.5))
print(verificar_recogida("A-101", 6))

## 4. *Function calling* automático

La forma más sencilla: le pasamos a Gemini la lista de funciones y la librería se encarga de todo. El modelo decide qué función llamar, la librería la ejecuta y le devuelve el resultado al modelo para que redacte la respuesta.

In [ ]:
SYSTEM_AGENTE = f"""Eres Nexo, el agente de atención de la {COOPERATIVA['nombre']}.
Usa tus herramientas para obtener datos y hacer cálculos: NUNCA inventes cifras ni calcules de memoria.
Reglamento:
""" + "\n".join(REGLAMENTO)

config_auto = types.GenerateContentConfig(system_instruction=SYSTEM_AGENTE, tools=HERRAMIENTAS, temperature=0)

r = llamar_con_reintentos(client.models.generate_content, model=MODELO,
                          contents="Soy el asociado A-101. Entregué 14 cargas con 13,5 % de humedad. ¿Cuánto me pagan?",
                          config=config_auto)
print(r.text)

# ¿Qué herramientas usó? El historial automático guarda cada llamada:
for contenido in r.automatic_function_calling_history or []:
    for parte in contenido.parts or []:
        if parte.function_call:
            print("🔧 Llamó a:", parte.function_call.name, dict(parte.function_call.args))

Funciona, pero es una "caja negra": no vemos bien cómo decide. Para **supervisar** un agente necesitamos ver cada paso. Vamos a programar el bucle nosotros mismos.

## 5. El bucle ReAct paso a paso

**ReAct** (*Reason + Act*; Yao et al., 2023) alterna tres momentos:

1. 🧠 **Razonar:** el modelo decide qué necesita.
2. 🔧 **Actuar:** pide ejecutar una herramienta con ciertos argumentos.
3. 👁️ **Observar:** recibe el resultado y vuelve a razonar.

El ciclo termina cuando el modelo responde sin pedir más herramientas, o cuando llegamos a un **máximo de pasos** (un límite de seguridad para que el agente no quede dando vueltas).

In [ ]:
# Diccionario nombre → función, para poder ejecutar la herramienta que pida el modelo.
MAPA_HERRAMIENTAS = {f.__name__: f for f in HERRAMIENTAS}

config_manual = types.GenerateContentConfig(
    system_instruction=SYSTEM_AGENTE + "\nAntes de usar una herramienta, explica en una frase qué vas a hacer y por qué.",
    tools=HERRAMIENTAS,
    temperature=0,
    # Desactivamos la ejecución automática: ahora NOSOTROS ejecutamos cada herramienta.
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def agente_react(meta, max_pasos=6, mostrar=True):
    """Ejecuta el bucle ReAct y devuelve la respuesta final y la traza de pasos."""
    historial = [types.Content(role="user", parts=[types.Part(text=meta)])]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta = llamar_con_reintentos(client.models.generate_content, model=MODELO,
                                          contents=historial, config=config_manual)
        contenido = respuesta.candidates[0].content
        historial.append(contenido)                      # guardamos lo que dijo el modelo

        # 🧠 Razonamiento (texto que escribió el modelo en este paso)
        texto = " ".join(p.text for p in (contenido.parts or []) if p.text)
        if texto and mostrar:
            print(f"🧠 Paso {paso} · Razonamiento: {texto.strip()}")

        llamadas = respuesta.function_calls or []
        if not llamadas:                                 # no pidió herramientas → terminó
            return texto, traza

        respuestas_herramientas = []
        for llamada in llamadas:
            argumentos = dict(llamada.args or {})
            funcion = MAPA_HERRAMIENTAS.get(llamada.name)
            # 🔧 Acción
            resultado = funcion(**argumentos) if funcion else {"error": "herramienta desconocida"}
            if mostrar:
                print(f"🔧 Paso {paso} · Acción: {llamada.name}({argumentos})")
                print(f"👁️ Paso {paso} · Observación: {resultado}")
            traza.append({"paso": paso, "herramienta": llamada.name, "argumentos": argumentos, "resultado": resultado})
            respuestas_herramientas.append(types.Part.from_function_response(name=llamada.name, response={"resultado": resultado}))
        # Devolvemos las observaciones al modelo para el siguiente paso
        historial.append(types.Content(role="user", parts=respuestas_herramientas))

    return "⚠️ Se alcanzó el máximo de pasos sin terminar. Escalar a una persona.", traza


final, traza = agente_react("Soy A-101, tengo 12 cargas listas. Si se puede, quiero que me las recojan en camión.")
print("\n✅ Respuesta final:", final)

Observa la traza: Nexo debería **verificar** la recogida antes de **registrarla**, tal como dice el docstring de `registrar_solicitud_recogida`. Probemos un caso que **no** cumple el reglamento (Rosa vive en una vereda sin vía carreteable):

In [ ]:
final, traza = agente_react("Soy A-102 y tengo 25 cargas. Mándenme el camión mañana, por favor.")
print("\n✅ Respuesta final:", final)
print("Solicitudes registradas hasta ahora:", SOLICITUDES)

> **Para tu bitácora:** copia una traza completa (razonamiento, acción y observación) y explica si cada decisión del agente fue correcta. Esta traza es el tipo de evidencia que mostrarás en la parte "detrás de cámaras" de tu video.

## 6. Planificar para alcanzar metas

Cuando la meta es grande, conviene que el agente primero haga un **plan** (lista de pasos) y luego lo ejecute. Esto permite **revisar el plan antes de actuar**, una forma sencilla de supervisión. Pedimos el plan con una salida estructurada:

In [ ]:
from pydantic import BaseModel, Field

class Paso(BaseModel):
    numero: int
    accion: str = Field(description="Qué hacer en este paso")
    herramienta: str = Field(description="Herramienta a usar, o 'ninguna'")

class Plan(BaseModel):
    meta: str
    pasos: list[Paso]

meta = ("Esta semana hay que organizar las recogidas de A-101 (15 cargas), A-102 (12 cargas) y A-103 (11 cargas). "
        "Registra solo las que cumplan el reglamento y explica a cada asociado el resultado.")

config_plan = types.GenerateContentConfig(
    system_instruction=SYSTEM_AGENTE + "\nHerramientas disponibles: " + ", ".join(MAPA_HERRAMIENTAS),
    response_mime_type="application/json", response_schema=Plan, temperature=0)
r = llamar_con_reintentos(client.models.generate_content, model=MODELO,
                          contents="Haz un plan paso a paso para esta meta (no la ejecutes): " + meta, config=config_plan)
plan = r.parsed or Plan.model_validate_json(r.text)
for p in plan.pasos:
    print(f"{p.numero}. {p.accion}  →  🔧 {p.herramienta}")

Una persona revisa el plan. Si está bien, el agente lo ejecuta con el bucle ReAct:

In [ ]:
plan_texto = "\n".join(f"{p.numero}. {p.accion}" for p in plan.pasos)
final, traza = agente_react(f"Meta: {meta}\nSigue este plan aprobado:\n{plan_texto}", max_pasos=10)
print("\n✅ Resultado:", final)
print("\n📋 Solicitudes registradas:", SOLICITUDES)

✅ Resultado esperado según el reglamento: se registran **A-101** (15 cargas, vía carreteable) y **A-103** (11 cargas, vía carreteable), y se rechaza **A-102** (su vereda no tiene vía carreteable). Compara con lo que hizo el agente: **verificar al agente también es parte del trabajo**.

## 7. Reto para tu proyecto 🎯

1. Define **al menos tres herramientas** (funciones de Python con docstrings claros) para tu agente.
2. Ejecuta tu agente con `agente_react` en tres situaciones: una normal, una que el reglamento no permite y una con datos faltantes.
3. Guarda las trazas: las usarás en la sección "Construcción y pruebas" de tu documento y en el video.

**Lecturas de apoyo:** Wooldridge y Jennings (1995); Xi et al. (2025); Yao et al. (2023), *ReAct*; Qin et al. (2025), *Tool learning with foundation models*.

## 📄 ¿Cómo guardar este cuaderno en PDF con todas las salidas?

1. *Entorno de ejecución → Reiniciar y ejecutar todas* (o *Ejecutar todas*). Espera a que todas las celdas terminen.
2. Revisa que cada celda muestre su salida y que no haya errores en rojo.
3. *Archivo → Imprimir* y, en *Destino*, elige **Guardar como PDF**.
4. Para el `.ipynb`: *Archivo → Descargar → Descargar .ipynb*.